# MongoDB Document Recovery and Collection Rules

[![Open Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

Download this notebook, open Colab, and choose **File > Upload notebook**. The full draft course is distributed separately from the public Week 1 repository.


Five synthetic tickets have been exported. We will restore them elsewhere, then
find an error that a correct count and a correct date type both miss. Your change
repairs one document from the saved artifact. We also restore the collection's
rules and test whether they work.

Keep all work and your short recovery recommendation in this notebook. The file
you produce contains document data, not a complete Atlas project backup.
`mongodump` and `mongorestore` cover a broader logical recovery scope.

## Connection or Local Practice

Keep `USE_ATLAS = False` for local practice with `mongomock`, an in-memory library.
It can run this document recovery exercise and enforce a unique index, but cannot
enforce MongoDB's server-side schema validator. We label that difference below.
Package installation still requires internet access.

For Atlas, use your existing Free cluster and set the switch to `True`. Colab runs
on Google's computer, not your laptop. The next cell contacts ipify without your
database credentials and prints the runtime's IPv4 address. Add that `/32` address
to Atlas temporarily. **Add Current IP** in your laptop browser can add the wrong
address. Do not enable access from everywhere.

In [ ]:
%pip -q install "pymongo>=4.13,<5" "mongomock>=4.3,<5"

In [ ]:
from datetime import datetime, timezone
from getpass import getpass
from ipaddress import IPv4Address
import hashlib
import json
from pathlib import Path
from pprint import pprint
from urllib.request import urlopen
from uuid import uuid4

import mongomock
from bson import json_util
from bson.json_util import CANONICAL_JSON_OPTIONS
from pymongo import ASCENDING, DESCENDING, MongoClient
from pymongo.errors import DuplicateKeyError, OperationFailure
from pymongo.server_api import ServerApi

USE_ATLAS = False
DATABASE_SUFFIX = uuid4().hex[:8]
SOURCE_DB = f"cst4714_recovery_source_{DATABASE_SUFFIX}"
RESTORE_DB = f"cst4714_recovery_restore_{DATABASE_SUFFIX}"
EXPORT_FILE = Path(f"/tmp/cst4714_tickets_{DATABASE_SUFFIX}.json")
print("Source:", SOURCE_DB)
print("Restore target:", RESTORE_DB)

if USE_ATLAS:
    try:
        with urlopen("https://api.ipify.org", timeout=10) as response:
            runtime_ip = str(IPv4Address(response.read().decode().strip()))
        print("Temporary Atlas IP access-list entry:", runtime_ip + "/32")
    except Exception:
        raise RuntimeError(
            "The runtime IP check failed. Retry or use local mode; "
            "do not open access from everywhere."
        ) from None

**Atlas only: pause before the next cell.** In **Network Access / IP Access List**,
add the printed address as a temporary entry and wait for it to become active.
Your database user needs read/write access, collection creation, index creation,
and `collMod` permission for the two printed practice databases. That user is
different from your Atlas website login. Ask for help scoping these permissions
rather than weakening an existing collection or a shared project.

In **Connect > Drivers**, choose Python and copy the driver URI. Replace its
password placeholder and percent-encode reserved password characters. Enter the
finished URI only into the hidden prompt. The code keeps certificate verification
enabled and clears the URI variable afterward. A successful `ping` proves a
response, not permission for every later operation.

Run configuration once per experiment, and clean up before starting a fresh run.
The data cells may be repeated in order. They reset only the `tickets` collections
inside these newly named practice databases, never a project collection.

In [ ]:
client = None
mongodb_uri = None
if USE_ATLAS:
    try:
        mongodb_uri = getpass("Atlas driver URI (hidden): ")
        client = MongoClient(
            mongodb_uri, tls=True, tlsInsecure=False, tz_aware=True,
            server_api=ServerApi("1", strict=True, deprecation_errors=True),
            serverSelectionTimeoutMS=10000, timeoutMS=10000,
        )
        client.admin.command("ping")
        print("MongoDB responded to ping.")
    except Exception:
        if client is not None:
            client.close()
        raise RuntimeError(
            "Atlas connection failed. Check the runtime IP rule, database "
            "credentials, driver URI, deployment state, and DNS/TLS. "
            "Keep certificate verification enabled; local mode is available."
        ) from None
    finally:
        mongodb_uri = None
else:
    client = mongomock.MongoClient(tz_aware=True)
    print("Local document recovery. Server validation will not execute.")

## 1. Save Five Tickets and Recover Them Elsewhere

The source collection has a focused validator in Atlas and a compound index. The
local library supports the documents and indexes but not server-side validation.
The unique `ticket_id` index prevents two documents from claiming the same ticket
number. The compound index supports the status-and-date workload from Week 11.
Neither index is part of an ordinary ticket document.

In [ ]:
source_database = client[SOURCE_DB]
# Reset only this notebook's source collection when repeating its setup.
source_database.drop_collection("tickets")

ticket_validator = {
    "$jsonSchema": {
        "bsonType": "object",
        "required": ["ticket_id", "status", "priority", "subject", "opened_at"],
        "properties": {
            "ticket_id": {"bsonType": ["int", "long"]},
            "status": {"enum": ["new", "open", "in_progress", "resolved", "closed"]},
            "priority": {"enum": ["low", "medium", "high", "urgent"]},
            "subject": {"bsonType": "string"},
            "opened_at": {"bsonType": "date"},
        },
    }
}

if USE_ATLAS:
    source_database.create_collection(
        "tickets", validator=ticket_validator,
        validationLevel="strict", validationAction="error",
    )
else:
    source_database.create_collection("tickets")
    print("Offline path: server-side $jsonSchema validation is not implemented by mongomock.")

source_tickets = source_database["tickets"]
source_tickets.create_index("ticket_id", unique=True, name="unique_ticket_id")
source_tickets.create_index(
    [("status", ASCENDING), ("opened_at", DESCENDING)], name="status_by_date"
)

source_documents = [
    {"ticket_id": 1001, "status": "open", "priority": "high",
     "subject": "Streetlight dark near bus stop",
     "opened_at": datetime(2026, 2, 1, 23, 10, tzinfo=timezone.utc)},
    {"ticket_id": 1002, "status": "in_progress", "priority": "medium",
     "subject": "Missed recycling pickup",
     "opened_at": datetime(2026, 2, 2, 15, 45, tzinfo=timezone.utc)},
    {"ticket_id": 1003, "status": "resolved", "priority": "urgent",
     "subject": "Low water pressure",
     "opened_at": datetime(2026, 2, 3, 12, 5, tzinfo=timezone.utc)},
    {"ticket_id": 1004, "status": "new", "priority": "low",
     "subject": "Broken bench slat",
     "opened_at": datetime(2026, 2, 4, 17, 20, tzinfo=timezone.utc)},
    {"ticket_id": 1005, "status": "resolved", "priority": "high",
     "subject": "Overflowing corner bin",
     "opened_at": datetime(2026, 2, 5, 14, 0, tzinfo=timezone.utc)},
]
source_tickets.insert_many(source_documents)

print("Source count:", source_tickets.count_documents({}))
print("Source indexes:", sorted(index["name"] for index in source_tickets.list_indexes()))
pprint(list(source_tickets.find({}, {"_id": 0}).sort("ticket_id", 1)))

### The Document File and Its Limits

Canonical Extended JSON preserves BSON type information such as dates and ObjectId
values in a JSON-compatible representation. File size and SHA-256 identify the
exact artifact; the separate restore determines whether it can be used. The
manifest printed below describes the expected collection and its rules. It stays
in the notebook, separately from the document file. That distinction matters if
someone gives you the JSON file alone.

`json.loads` recognizes JSON syntax but leaves `$date` as a dictionary.
`json_util.loads` understands MongoDB's type markers. A UTC datetime and a string
containing a date are different stored values. For this fixture, BSON stores
dates to millisecond precision. The same stored date may display in another
timezone without representing a different instant.

In [ ]:
documents_to_export = list(source_tickets.find({}).sort("ticket_id", ASCENDING))
JSON_OPTIONS = CANONICAL_JSON_OPTIONS.with_options(tz_aware=True)
expected_canonical = json_util.dumps(
    documents_to_export, json_options=JSON_OPTIONS, sort_keys=True
)
export_text = json_util.dumps(
    documents_to_export,
    json_options=JSON_OPTIONS,
    indent=2,
)
EXPORT_FILE.write_text(export_text + "\n", encoding="utf-8")

export_bytes = EXPORT_FILE.read_bytes()
export_sha256 = hashlib.sha256(export_bytes).hexdigest()
print("Artifact:", EXPORT_FILE)
print("Bytes:", len(export_bytes))
print("SHA-256:", export_sha256)
print("First 300 characters:\n", export_text[:300])

manifest = {
    "collection": "tickets",
    "document_count": 5,
    "ticket_ids": [1001, 1002, 1003, 1004, 1005],
    "sha256": export_sha256,
    "indexes": source_tickets.index_information(),
    "validator": ticket_validator,
    "server_validator_installed": USE_ATLAS,
}
pprint(manifest)
plain_json = json.loads(export_text)
print("Plain JSON parser gives:", type(plain_json[0]["opened_at"]).__name__)

### A Separate Restore Target

The restore database name is different from the source. Parsing with `json_util`
reconstructs BSON-aware Python values before insertion. We verify the file hash
and parse it **before** resetting the disposable restore collection. A hash match
means these bytes match the recorded digest; it does not prove that the export
was complete, came from a trustworthy source, or represents a consistent live
multi-collection moment. These five source documents are not changing during export.

In [ ]:
restore_bytes = EXPORT_FILE.read_bytes()
assert hashlib.sha256(restore_bytes).hexdigest() == manifest["sha256"]
restored_documents = json_util.loads(restore_bytes.decode("utf-8"), json_options=JSON_OPTIONS)
assert len(restored_documents) == manifest["document_count"]
assert [doc["ticket_id"] for doc in restored_documents] == manifest["ticket_ids"]
parsed_canonical = json_util.dumps(
    restored_documents, json_options=JSON_OPTIONS, sort_keys=True
)
assert parsed_canonical == expected_canonical

restore_database = client[RESTORE_DB]
assert RESTORE_DB != SOURCE_DB
restore_database.drop_collection("tickets")
restore_tickets = restore_database["tickets"]
restore_result = restore_tickets.insert_many(restored_documents)
print("Restored documents:", len(restore_result.inserted_ids))
print("Restore count:", restore_tickets.count_documents({}))

### A Verification Baseline

The expected active IDs are 1001, 1002, and 1004. The other tickets are resolved,
not missing. We compare complete values as well as count, identifiers, and types.
The full comparison serializes values with the same Canonical Extended JSON
options and sorted field names. This also distinguishes an integer from a double
that displays the same numeric value. For this small fixture we can inspect every
saved document; a large production
dataset needs a verification strategy appropriate to its size and invariants.

In [ ]:
source_ids = [
    doc["ticket_id"]
    for doc in source_tickets.find({}, {"_id": 0, "ticket_id": 1}).sort("ticket_id", 1)
]
restore_ids = [
    doc["ticket_id"]
    for doc in restore_tickets.find({}, {"_id": 0, "ticket_id": 1}).sort("ticket_id", 1)
]
print("Source IDs:", source_ids)
print("Restore IDs:", restore_ids)
assert source_ids == restore_ids == manifest["ticket_ids"]

restored_sample = restore_tickets.find_one({"ticket_id": 1001})
print("Restored opened_at type:", type(restored_sample["opened_at"]).__name__)
assert isinstance(restored_sample["opened_at"], datetime)

active = list(
    restore_tickets.find(
        {"status": {"$in": ["new", "open", "in_progress"]}},
        {"_id": 0, "ticket_id": 1, "status": 1},
    ).sort("ticket_id", 1)
)
print("Active restored tickets:", active)
assert [doc["ticket_id"] for doc in active] == [1001, 1002, 1004]
restored_canonical = json_util.dumps(
    list(restore_tickets.find({}).sort("ticket_id", 1)),
    json_options=JSON_OPTIONS, sort_keys=True,
)
assert restored_canonical == expected_canonical
print("All five restored documents match the saved values.")

## 2. Diagnose and Repair a Misleading Restore

A migration after restoration replaced one ticket's subject with incorrect text.
The mistake still satisfies the schema: the subject remains a string. Predict
which checks below will notice it. Choose `1001` or `1004` in the next cell.
This deliberately changes the disposable **restore target**, not the source or
the saved artifact. You are testing a recovery check, not simulating replication.

In [ ]:
DAMAGED_TICKET = 1001  # You may choose 1004 instead.
assert DAMAGED_TICKET in (1001, 1004)
restore_tickets.update_one(
    {"ticket_id": DAMAGED_TICKET},
    {"$set": {"subject": "Text replaced by an incorrect migration"}},
)

current_documents = list(restore_tickets.find({}).sort("ticket_id", 1))
print("Correct count:", len(current_documents) == manifest["document_count"])
print("Correct IDs:", [doc["ticket_id"] for doc in current_documents] == manifest["ticket_ids"])
print(
    "All dates are datetime:",
    all(isinstance(doc["opened_at"], datetime) for doc in current_documents),
)
current_canonical = json_util.dumps(current_documents, json_options=JSON_OPTIONS, sort_keys=True)
print("All document values match:", current_canonical == expected_canonical)

# Build the lookup from documents parsed from the verified artifact, not a new source query.
saved_by_id = {doc["ticket_id"]: doc for doc in restored_documents}
print("Saved subject:", saved_by_id[DAMAGED_TICKET]["subject"])
print("Restored subject:", restore_tickets.find_one({"ticket_id": DAMAGED_TICKET})["subject"])

### Your Repair

The current database is wrong, so use the saved version as the recovery source.
Replace `None` below with `saved_by_id[DAMAGED_TICKET]`. Read the two arguments
to `replace_one`: the first selects one ticket, and the second is its complete
replacement document. The saved `_id` still identifies that same document.

Run the cell and verify that all five documents match again. Rerunning this
repair should not insert a sixth ticket. Explain why the first three checks
above passed even when the restored subject was wrong. If you want to repeat
with the other ticket, repair the first one before making another change.

In [ ]:
recovery_document = None  # Replace None with the saved document described above.

if recovery_document is None:
    print("Repair not completed: choose the saved document, then rerun this cell.")
else:
    assert recovery_document["ticket_id"] == DAMAGED_TICKET
    repair = restore_tickets.replace_one({"ticket_id": DAMAGED_TICKET}, recovery_document)
    assert repair.matched_count == 1

repaired_canonical = json_util.dumps(
    list(restore_tickets.find({}).sort("ticket_id", 1)),
    json_options=JSON_OPTIONS, sort_keys=True,
)
all_values_recovered = repaired_canonical == expected_canonical
print("All document values recovered:", all_values_recovered)
print("Ticket count:", restore_tickets.count_documents({}))

## 3. Restore Rules and Test Their Behavior

Collection indexes and validators are database metadata. The document-only export
did not recreate them automatically. MongoDB creates the `_id_` index itself.
Our separate setup instructions, not the JSON document file, supply the unique
ticket key, compound index, and validator. Rerunning this rule cell is safe.

In [ ]:
print("Restore indexes before repair:", sorted(restore_tickets.index_information()))
print("On the first document-only restore, only the automatic _id_ index exists.")

restore_tickets.create_index("ticket_id", unique=True, name="unique_ticket_id")
restore_tickets.create_index(
    [("status", ASCENDING), ("opened_at", DESCENDING)], name="status_by_date"
)

if USE_ATLAS:
    restore_database.command(
        "collMod",
        "tickets",
        validator=ticket_validator,
        validationLevel="strict",
        validationAction="error",
    )
    print("Recreated index and server-side validator in the restore target.")
else:
    print("Recreated index. Offline path records, but cannot enforce, the server validator.")

print("Restore indexes after repair:", sorted(restore_tickets.index_information()))

### Allowed and Rejected Writes

First a valid test document must succeed. Then a duplicate ticket number must
fail, even with a different `_id`. Atlas additionally tests an invalid status,
a string date, and a missing date. Error 121 means document validation failed;
error 11000 means a unique index rejected a duplicate. Other errors are not
evidence that these rules worked. All temporary test documents are removed.

Local mode tests the unique index, but its schema results are a **supplied trace
to interpret**, not results from your runtime. A validator also cannot detect
every wrong value: the damaged subject above was still a permitted string.

In [ ]:
valid_document = {
    "ticket_id": 1099,
    "status": "new",
    "priority": "low",
    "subject": "Temporary allowed write",
    "opened_at": datetime(2026, 2, 6, tzinfo=timezone.utc),
}
test_ids = [1099, 1100, 1101, 1102]
try:
    restore_tickets.insert_one(valid_document)
    print("Valid document accepted.")
    duplicate = dict(saved_by_id[1001])
    duplicate.pop("_id")  # Different internal ID, same ticket_id.
    try:
        duplicate_result = restore_tickets.insert_one(duplicate)
    except DuplicateKeyError:
        print("Duplicate ticket_id rejected by the unique index: 11000.")
    else:
        restore_tickets.delete_one({"_id": duplicate_result.inserted_id})
        raise AssertionError("The unique ticket_id index did not reject a duplicate.")

    cases = [
        ("Invalid status", {**valid_document, "ticket_id": 1100, "status": "almost_done"}),
        ("Date stored as text", {**valid_document, "ticket_id": 1101, "opened_at": "2026-02-06"}),
        ("Missing date", {
            key: value for key, value in valid_document.items() if key != "opened_at"
        }),
    ]
    cases[2][1]["ticket_id"] = 1102
    for label, document in cases:
        document.pop("_id", None)  # insert_one added an _id to valid_document.
        if USE_ATLAS:
            try:
                restore_tickets.insert_one(document)
            except OperationFailure as error:
                if error.code != 121:
                    raise
                print(label, "rejected. MongoDB error code:", error.code)
            else:
                raise AssertionError(label + " unexpectedly accepted.")
        else:
            print(
                "Supplied trace, NOT a result from this runtime:",
                label, "would be rejected with code 121.",
            )
finally:
    restore_tickets.delete_many({"ticket_id": {"$in": test_ids}})

assert restore_tickets.count_documents({}) == 5
print("Temporary test documents removed; five practice tickets remain.")

### How This Compares With a Database Dump

| Concern | Canonical Extended JSON exercise | `mongodump` / `mongorestore` |
|---|---|---|
| selected document values | yes | yes |
| BSON type representation | preserved through Extended JSON when parsed correctly | native BSON archive |
| collection options/validator | not recreated automatically | collection metadata/options within documented behavior |
| index definitions | not recreated automatically | included in dump metadata |
| Atlas database users and network rules | no | no, managed separately |
| multi-collection point consistency | not established by this one-collection script | depends on topology, options, and documented tool behavior |

For an Atlas Free database-level backup, use current compatible MongoDB Database
Tools and the documented `mongodump`/`mongorestore` process. This notebook teaches
the recovery sequence and the limitations of a narrower artifact.

## Your Recovery Recommendation

If a colleague handed you only the JSON file, what could you recover, and what
else would you ask for before reopening the application? Write a short answer
using one actual result, one separately recreated rule or index, and a comparison
with Week 8's PostgreSQL archive. Include why the wrong subject passed the count
check and how you recovered it. Name whether server validation ran in Atlas or
you interpreted the supplied local trace. Do not claim to have tested failover.

The comparison table above is supplied reading, not another submission task.
Before submitting, remove any accidentally saved credential from source or output.

Replace this paragraph with your short recovery recommendation. This is the only
written response; keep it in the notebook with your completed repair and outputs.

## Cleanup

Run this cell after finishing. It removes only the two practice `tickets`
collections and this run's temporary JSON file, then closes the Python client.
Other collections are left alone. No exported file needs to be submitted.

If you used Atlas, remove the temporary IP access-list entry in the dashboard.
Closing a Python client does not shut down an Atlas cluster. Leave shared
deployments and other projects untouched. Remove the runtime-IP cell output
before submitting, along with any credential accidentally entered in source.

In [ ]:
source_database.drop_collection("tickets")
restore_database.drop_collection("tickets")
EXPORT_FILE.unlink(missing_ok=True)
client.close()
print("Removed both practice collections and the temporary file; client closed.")

## Further Reading

- [PyMongo Extended JSON](https://www.mongodb.com/docs/languages/python/pymongo-driver/current/data-formats/extended-json/)
- [MongoDB schema validation](https://www.mongodb.com/docs/manual/core/schema-validation/)
- [Atlas Free limits](https://www.mongodb.com/docs/atlas/reference/free-shared-limitations/)
- [mongodump](https://www.mongodb.com/docs/database-tools/mongodump/) and [mongorestore](https://www.mongodb.com/docs/database-tools/mongorestore/)
- [Atlas IP access list](https://www.mongodb.com/docs/atlas/security/ip-access-list/)

Course prose: CC BY-NC-SA 4.0. Code: MIT. Synthetic fixture: CC0.